In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
sys.path.append(str(PROJECT_ROOT / "spark"))

In [2]:
#import relevant packages
import sys
import os
import argparse 
from pathlib import Path
from pyspark.sql import SparkSession    
from pyspark.sql.functions import col

from spark.framework.iceberg.schema_loader import load_platform_schema
#from spark.framework.iceberg.iceberg_utils import (ensure_namespace_exists)
from spark.framework.iceberg.table_manager import create_table_if_not_exists, ensure_namespace_exists, write_to_iceberg
#fetch the project root directory 
#PROJECT_ROOT = Path(__file__).resolve().parents[1]
#sys.path.append(str(PROJECT_ROOT))
from spark.common.config_loader import load_config
from spark.framework.metadata.entity_config_loader import (load_entity_config, )
from spark.framework.transformation.silver_builder import (build_silver)
from spark.framework.metadata.schema_generator import generate_schema

from pyspark.sql.types import (
    StructType,
    StructField,
    BinaryType,
    StringType,
    IntegerType,
    LongType,
    TimestampType
)
from spark.framework.spark.spark_session import create_spark_session

BRONZE_SCHEMA = StructType([
    StructField("key", BinaryType(), True),
    StructField("value", BinaryType(), True),
    StructField("topic", StringType(), False),
    StructField("partition", IntegerType(), False),
    StructField("offset", LongType(), False),
    StructField("timestamp", TimestampType(), True),
    StructField("timestampType", StringType(), True)
])

Creating Spark Session...


In [3]:
entity_name = "customer"

    ##Read environment variable for ENV, default to "local" if not set
env = os.getenv("ENV", "local")
print(f"Environment: {env}")

print(f"Starting Silver Stream for Entity: {entity_name}")

    #Load configuration for environment and entity
env_config = load_config(env)
entity_config = load_entity_config(entity_name)

    #silver schema path
silver_schema_path = PROJECT_ROOT / "configs" / "entities" / f"{entity_name}.yaml"

    #create the bronze table and silver table names using the entity name from the entity configuration
catalog_name = env_config['iceberg']['catalog_name']
bronze_table = f"{catalog_name}.bronze.{entity_config['entity_name']}"
print(f"Bronze Table: {bronze_table}")
silver_table = f"{catalog_name}.silver.{entity_config['entity_name']}"
print(f"Silver Table: {silver_table}")
    
    #create a checkpoint path for the silver stream using the entity name from the entity configuration
checkpoint_path = (
    f"{env_config['storage']['checkpoint_root_path']}/silver/{entity_config['entity_name']}"
    )
print(f"Checkpoint Path: {checkpoint_path}") 

Environment: local
Starting Silver Stream for Entity: customer
Bronze Table: insightflow.bronze.customer
Silver Table: insightflow.silver.customer
Checkpoint Path: gs://insightflowai-data-prod/checkpoints/silver/customer


In [4]:
 #create Spark Session
spark = create_spark_session("Silver Stream", env)
        
print("1. Spark Session Created")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/08 11:34:25 WARN Utils: Your hostname, Sauravs-MacBook-Air.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.5 instead (on interface en0)
26/07/08 11:34:25 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
:: loading settings :: url = jar:file:/Users/sauravpandey/Projects/streaming/subscription-platform/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /Users/sauravpandey/.ivy2.5.2/cache
The jars for the packages stored in: /Users/sauravpandey/.ivy2.5.2/jars
org.apache.spark#spark-sql-kafka-0-10_2.13 added as a dependency
com.google.cloud.bigdataoss#gcs-connector added as a dependency
org.apache.iceberg#iceberg-spark-runtime-4.1_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-90dd26e1-2e32-409e-8bf7-10c1131d8dad;1.0
	confs: [default]
	foun

1. Spark Session Created


In [5]:
spark.sql("DROP TABLE insightflow.silver.customer")

DataFrame[]

In [6]:
#Crete bronze schema to allow spark to read the bronze parquet files and convert to string format for silver layer processing
silver_schema = load_platform_schema(silver_schema_path)  

silver_schema

{'schema_version': 1,
 'entity_name': 'customer',
 'source': {'topic': 'insightflow.public.customer'},
 'business_keys': 'customer_id',
 'columns': {'customer_id': {'datatype': 'string',
   'nullable': False,
   'business_key': True},
  'customer_name': {'datatype': 'string', 'nullable': False},
  'customer_status': {'datatype': 'string', 'nullable': False},
  'industry': {'datatype': 'string', 'nullable': True},
  'customer_size': {'datatype': 'string', 'nullable': True},
  'customer_tier': {'datatype': 'string', 'nullable': True},
  'customer_start_date': {'datatype': 'date',
   'nullable': True,
   'source_format': 'epoch_days'},
  'customer_end_date': {'datatype': 'date',
   'nullable': True,
   'source_format': 'epoch_days'},
  'created_timestamp': {'datatype': 'timestamp',
   'nullable': False,
   'source_format': 'epoch_micros'},
  'updated_timestamp': {'datatype': 'timestamp',
   'nullable': False,
   'source_format': 'epoch_micros'}},
 'system_columns': {'op': {'datatype': 'st

In [7]:
print("entity_config: ", entity_config)
print("silver_schema: ", silver_schema)

entity_config:  {'schema_version': 1, 'entity_name': 'customer', 'source': {'topic': 'insightflow.public.customer'}, 'business_keys': 'customer_id', 'columns': {'customer_id': {'datatype': 'string', 'nullable': False, 'business_key': True}, 'customer_name': {'datatype': 'string', 'nullable': False}, 'customer_status': {'datatype': 'string', 'nullable': False}, 'industry': {'datatype': 'string', 'nullable': True}, 'customer_size': {'datatype': 'string', 'nullable': True}, 'customer_tier': {'datatype': 'string', 'nullable': True}, 'customer_start_date': {'datatype': 'date', 'nullable': True, 'source_format': 'epoch_days'}, 'customer_end_date': {'datatype': 'date', 'nullable': True, 'source_format': 'epoch_days'}, 'created_timestamp': {'datatype': 'timestamp', 'nullable': False, 'source_format': 'epoch_micros'}, 'updated_timestamp': {'datatype': 'timestamp', 'nullable': False, 'source_format': 'epoch_micros'}}, 'system_columns': {'op': {'datatype': 'string', 'nullable': False}, 'ts_ms': {

In [8]:
entity_schema = generate_schema(entity_config)
print("Entity Schema: ", entity_schema)

Entity Schema:  StructType([StructField('customer_id', StringType(), False), StructField('customer_name', StringType(), False), StructField('customer_status', StringType(), False), StructField('industry', StringType(), True), StructField('customer_size', StringType(), True), StructField('customer_tier', StringType(), True), StructField('customer_start_date', IntegerType(), True), StructField('customer_end_date', IntegerType(), True), StructField('created_timestamp', LongType(), False), StructField('updated_timestamp', LongType(), False)])


In [9]:
entity_schema = generate_schema(entity_config)
print("Entity Schema: ", entity_schema)

Entity Schema:  StructType([StructField('customer_id', StringType(), False), StructField('customer_name', StringType(), False), StructField('customer_status', StringType(), False), StructField('industry', StringType(), True), StructField('customer_size', StringType(), True), StructField('customer_tier', StringType(), True), StructField('customer_start_date', IntegerType(), True), StructField('customer_end_date', IntegerType(), True), StructField('created_timestamp', LongType(), False), StructField('updated_timestamp', LongType(), False)])


In [10]:
    #ensure the silver namespace exists in the iceberg catalog, if not create it
ensure_namespace_exists(spark, catalog_name, "silver")



In [11]:
    #create the silver table in the iceberg catalog if it does not exist, using the silver schema and the silver path
create_table_if_not_exists(
    spark=spark,
    catalog=catalog_name,
    namespace="silver",
    table_name=entity_name,
    schema=silver_schema
    )

TypeError: create_table_if_not_exists() missing 1 required positional argument: 'logger'

In [12]:
 #Read Bronze data from the bronze storage path in parquet format and convert to string format for Silver layer processing
bronze_df = (
        spark.read
        .table(bronze_table)
    )
print("2. Bronze Stream Created\n",bronze_df.show(10, truncate=False))

+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [12]:
 #Read Bronze data from the bronze storage path in parquet format and convert to string format for Silver layer processing
bronze_df = (
    spark.readStream
    .table(bronze_table)
)
print("2. Bronze Stream Created\n",bronze_df)

2. Bronze Stream Created
 DataFrame[key: binary, value: binary, topic: string, partition: int, offset: bigint, timestamp: timestamp, timestampType: int]


In [13]:
from framework.logging.logger import get_logger
#configure logging
logger = get_logger(
    "silver_Stream",
    env_config
    )

In [14]:
#call silver builder function to process the bronze data and and transform for silver layer processing and write to silver storage path in parquet format
valid_df, invalid_df = build_silver(bronze_df, silver_schema, env_config, entity_name, logger=logger)
print("Silver DF Built")

2026-07-08 11:36:14,077 | INFO     | silver_Stream | Starting Silver Builder...
2026-07-08 11:36:14,080 | INFO     | silver_Stream | Bronze Data Read Successfully from stream file
2026-07-08 11:36:14,081 | INFO     | silver_Stream | Silver Data will be written to: gs://insightflowai-data-prod/silver/customer
2026-07-08 11:36:14,081 | INFO     | silver_Stream | Extracting Required Fields for Silver Layer Processing...
2026-07-08 11:36:14,126 | INFO     | silver_Stream | calling parse_debezium function to extract before, after, op, source, ts_ms from raw_payload
Running Debezium Parser
2026-07-08 11:36:14,151 | INFO     | silver_Stream | Debezium Parser Completed
calling map_entity function to convert cdc event into entity-specific silver records
2026-07-08 11:36:14,151 | INFO     | silver_Stream | Running Entity Mapper
Running Normalizer
2026-07-08 11:36:14,218 | INFO     | silver_Stream | Normalizer Completed
2026-07-08 11:36:14,219 | INFO     | silver_Stream | Entity Mapping Completed

In [17]:
silver_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_status: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- customer_size: string (nullable = true)
 |-- customer_tier: string (nullable = true)
 |-- customer_start_date: date (nullable = true)
 |-- customer_end_date: date (nullable = true)
 |-- created_timestamp: timestamp (nullable = true)
 |-- updated_timestamp: timestamp (nullable = true)
 |-- op: string (nullable = true)
 |-- ts_ms: timestamp (nullable = true)
 |-- topic: string (nullable = false)
 |-- partition: integer (nullable = false)
 |-- offset: long (nullable = false)
 |-- timestamp: timestamp (nullable = true)



In [16]:
silver_df.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- customer_status: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- customer_size: string (nullable = true)
 |-- customer_tier: string (nullable = true)
 |-- customer_start_date: date (nullable = true)
 |-- customer_end_date: date (nullable = true)
 |-- created_timestamp: timestamp (nullable = true)
 |-- updated_timestamp: timestamp (nullable = true)
 |-- op: string (nullable = true)
 |-- ts_ms: timestamp (nullable = true)
 |-- topic: string (nullable = false)
 |-- partition: integer (nullable = false)
 |-- offset: long (nullable = false)
 |-- timestamp: timestamp (nullable = true)



In [18]:
silver_df = silver_df.drop(
    "topic",
    "partition",
    "offset",
    "timestamp"
)

In [16]:
silver_df

DataFrame[customer_id: string, customer_name: string, customer_status: string, industry: string, customer_size: string, customer_tier: string, customer_start_date: date, customer_end_date: date, created_timestamp: timestamp, updated_timestamp: timestamp, op: string, ts_ms: timestamp]

In [19]:
 #write to silver storage path in parquet format with metadata and payload fields extracted from the bronze layer parquet data
query = (
    silver_df.writeStream
      .foreachBatch(
            lambda batch_df, batch_id:
                write_to_iceberg(
                    batch_df=batch_df,
                    batch_id=batch_id,
                    table_name=silver_table,
                    mode="append"
                    
                )
      )
      .option("mergeSchema", "true")
      .option(
          "checkpointLocation",
          checkpoint_path
      )
      .start()
)

26/07/05 00:34:10 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [24]:
query.isActive

False

In [23]:
query.lastProgress

{
    "id": "77d4506e-e344-49d1-9240-2f9c9d996852",
    "runId": "feff19e3-4c36-447a-bb1f-894b223d1d3e",
    "name": null,
    "timestamp": "2026-07-04T19:07:07.524Z",
    "batchId": 1,
    "batchDuration": 7796,
    "numInputRows": 0,
    "inputRowsPerSecond": 0.0,
    "processedRowsPerSecond": 0.0,
    "durationMs": {
        "latestOffset": 7795,
        "triggerExecution": 7796
    },
    "stateOperators": [],
    "sources": [
        {
            "description": "org.apache.iceberg.spark.source.SparkMicroBatchStream@1731a3e4",
            "startOffset": {
                "version": 1,
                "snapshot_id": 7660280243483491480,
                "position": 1,
                "scan_all_files": false
            },
            "endOffset": {
                "version": 1,
                "snapshot_id": 7660280243483491480,
                "position": 1,
                "scan_all_files": false
            },
            "latestOffset": null,
            "numInputRows": 0,
     

In [1]:
for q in spark.streams.active:
    print(q.name)
    print(q.id)
    print(q.status)

NameError: name 'spark' is not defined

In [18]:
for q in spark.streams.active:
    q.stop()

ConnectionRefusedError: [Errno 61] Connection refused

In [80]:
import inspect
from spark.framework.transformation.normalizer import normalize

print(inspect.getsource(normalize))

def normalize(
        mapped_df: DataFrame,
        entity_config: dict
    ) -> DataFrame:
    print("Running Normalizer")
    """ Normalize the mapped DataFrame based on the entity configuration """

    for column_name, metadata in entity_config["columns"].items():
        print(entity_config["columns"].keys())
        print(f"Column: {column_name}")
        print(f"source_format: '{source_format}'")
        print(f"type: {type(source_format)}")
        print(f"repr: {repr(source_format)}")
        if "source_format" in metadata:
            source_format = metadata["source_format"]
            if source_format == "epoch_micros":
                mapped_df = mapped_df.withColumn(
                    column_name,
                    timestamp_micros(col(column_name))
                )
            elif source_format == "epoch_days":
                mapped_df = mapped_df.withColumn(
                    column_name,
                    date_add(to_date(lit("1970-01-01")), col(column_nam

In [16]:
spark.sql("select * from insightflow.silver.customer").show(10, truncate=False)

+-----------+-------------+---------------+-------------+-------------+-------------+-------------------+-----------------+--------------------------+--------------------------+---+-------------------+
|customer_id|customer_name|customer_status|industry     |customer_size|customer_tier|customer_start_date|customer_end_date|created_timestamp         |updated_timestamp         |op |ts_ms              |
+-----------+-------------+---------------+-------------+-------------+-------------+-------------------+-----------------+--------------------------+--------------------------+---+-------------------+
|CUST1001   |OpenAI       |ACTIVE         |Technology   |Enterprise   |Platinum     |2026-07-03         |NULL             |2026-07-03 10:46:16.108056|2026-07-03 10:46:16.108056|c  |2026-07-03 10:46:16|
|CUST1002   |Microsoft    |ACTIVE         |Technology   |Enterprise   |Gold         |2026-07-03         |NULL             |2026-07-03 11:57:05.05155 |2026-07-03 11:57:05.05155 |c  |2026-07-03 

In [11]:
spark.read.table(
    "insightflow.bronze.customer"
).show()

+--------------------+--------------------+--------------------+---------+------+--------------------+-------------+
|                 key|               value|               topic|partition|offset|           timestamp|timestampType|
+--------------------+--------------------+--------------------+---------+------+--------------------+-------------+
|[7B 22 73 63 68 6...|[7B 22 73 63 68 6...|insightflow.publi...|        0|    12|2026-07-03 11:57:...|            0|
|[7B 22 73 63 68 6...|[7B 22 73 63 68 6...|insightflow.publi...|        0|    14|2026-07-03 13:09:...|            0|
|[7B 22 73 63 68 6...|[7B 22 73 63 68 6...|insightflow.publi...|        0|    12|2026-07-03 11:57:...|            0|
|[7B 22 73 63 68 6...|[7B 22 73 63 68 6...|insightflow.publi...|        0|    12|2026-07-03 11:57:...|            0|
|[7B 22 73 63 68 6...|[7B 22 73 63 68 6...|insightflow.publi...|        0|    11|2026-07-03 10:46:...|            0|
|[7B 22 73 63 68 6...|[7B 22 73 63 68 6...|insightflow.publi...|

In [12]:
bronze = spark.read.table("insightflow.bronze.customer")


In [13]:
bronze.selectExpr(
    "CAST(value AS STRING) AS raw_payload"
).show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [14]:
spark.read.table("insightflow.bronze.customer") \
    .selectExpr("CAST(value AS STRING) AS raw_payload") \
    .show(1, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------